# INC-007 Weekly Biostatistics Review

This notebook inspects an existing pipeline run against the INC-007 contract (`specs/INC-007.md`): the `gold_biostat_weekly` participant-week table, its week boundaries, coverage columns, the 4-valid-day availability threshold, its consistency with `gold_clinical_daily`, and its idempotent and incremental behavior. The supplied database is opened read-only. The rerun and incremental sections work on temporary copies. Raw inputs are read only.

Generate the run first with `python -m wearable_pipeline run --data-dir data --output-dir output/runs/inc007`. Set `INC007_RUN_DIR` if the output directory differs, `INC007_DATA_DIR` to override `data/`, and `INC007_PREVIOUS_RUN_DIR` (default `output/runs/inc005`, a run before the weekly table) to compare the pre-existing daily gold columns. Sections 8 and 9 run two full rebuilds and one incremental run (about 4 minutes).

## Review Sections
1. Environment, paths and display helpers.
2. Row counts, schema, key and week boundaries.
3. Coverage columns against the daily table.
4. Weekly values reconciled with `gold_clinical_daily`.
5. Weekly steps and heart-rate values recomputed independently from silver.
6. The availability threshold (`MIN_VALID_DAYS = 4`) and its use on this dataset.
7. Daily gold columns versus a previous run (optional).
8. Rerun on an isolated copy: identical tables and report; raw data unchanged.
9. One corrected record: incremental versus a full rebuild changes exactly one weekly row.
10. Acceptance summary with PASS/FAIL/NOT RUN.

Aggregates and a few participant-week keys are displayed; raw record values are not printed.

In [1]:
import hashlib
import html
import json
import os
import shutil
import sys
import tempfile
from pathlib import Path

import duckdb
from IPython.display import HTML, display

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'src' / 'wearable_pipeline').is_dir()
)
sys.path.insert(0, str(ROOT / 'src'))
from wearable_pipeline.compare import compare_outputs, format_comparison
from wearable_pipeline.incremental import run_incremental
from wearable_pipeline.pipeline import run_pipeline
from wearable_pipeline.weekly import MIN_VALID_DAYS, WEEKLY_COLUMNS

DATA_DIR = Path(os.environ.get('INC007_DATA_DIR', ROOT / 'data')).resolve()
RUN_DIR = Path(os.environ.get('INC007_RUN_DIR', ROOT / 'output' / 'runs' / 'inc007')).resolve()
PREVIOUS_RUN_DIR = Path(
    os.environ.get('INC007_PREVIOUS_RUN_DIR', ROOT / 'output' / 'runs' / 'inc005')
).resolve()
DATABASE_PATH = RUN_DIR / 'wearable.duckdb'
REPORT_PATH = RUN_DIR / 'quality' / 'report.md'

if not DATABASE_PATH.is_file():
    raise FileNotFoundError(f'DuckDB file not found: {DATABASE_PATH}. Set INC007_RUN_DIR.')
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f'Raw data directory not found: {DATA_DIR}')

connection = duckdb.connect(str(DATABASE_PATH), read_only=True)
acceptance = {}


def render_table(title, columns, rows):
    header = ''.join('<th>' + html.escape(str(column)) + '</th>' for column in columns)
    body = ''.join(
        '<tr>' + ''.join('<td>' + html.escape(str(value)) + '</td>' for value in row) + '</tr>'
        for row in rows
    )
    display(HTML(
        '<h4>' + html.escape(title) + '</h4><table><thead><tr>' + header
        + '</tr></thead><tbody>' + body + '</tbody></table>'
    ))


def show_query(title, sql, parameters=None, db=connection):
    cursor = db.execute(sql, parameters or [])
    render_table(title, [item[0] for item in cursor.description], cursor.fetchall())


def status(ok):
    return 'PASS' if ok else 'FAIL'


def tree_hash(path):
    digest = hashlib.sha256()
    for file in sorted(Path(path).rglob('*')):
        if file.is_file():
            digest.update(str(file.relative_to(path)).encode())
            digest.update(hashlib.sha256(file.read_bytes()).digest())
    return digest.hexdigest()


report_text = REPORT_PATH.read_text(encoding='utf-8')
run_id = next(
    (line.split(': ', 1)[1] for line in report_text.splitlines() if line.startswith('Run id: ')),
    None,
)
RAW_HASH_BEFORE = tree_hash(DATA_DIR)
print(f'Database: {DATABASE_PATH}')
print(f'Data directory: {DATA_DIR}')
print(f'Run ID: {run_id}')
print(f'Raw data fingerprint before: {RAW_HASH_BEFORE[:16]}')
print(f'DuckDB version: {duckdb.__version__}')

Database: /home/michaelmadeira/wearable-health-data-pipeline/output/runs/inc007/wearable.duckdb
Data directory: /home/michaelmadeira/wearable-health-data-pipeline/data
Run ID: a826c8397cf5ad8e
Raw data fingerprint before: abd71e9575b8ae2a
DuckDB version: 1.5.6


In [2]:
# Section 2: row counts, schema, key and week boundaries.
# 200 = 50 enrolled participants x 4 whole seven-day weeks in the default 28-day window.
weekly_rows = connection.execute('SELECT COUNT(*) FROM gold_biostat_weekly').fetchone()[0]
distinct_keys = connection.execute(
    'SELECT COUNT(DISTINCT (participant_id, week_id)) FROM gold_biostat_weekly').fetchone()[0]
enrolled = connection.execute(
    'SELECT COUNT(DISTINCT participant_id) FROM gold_clinical_daily').fetchone()[0]
daily_rows = connection.execute('SELECT COUNT(*) FROM gold_clinical_daily').fetchone()[0]
render_table(
    'Row counts',
    ['check', 'expected', 'observed', 'status'],
    [
        ('gold_biostat_weekly rows', enrolled * 4, weekly_rows, status(weekly_rows == enrolled * 4 == 200)),
        ('distinct (participant_id, week_id) keys', weekly_rows, distinct_keys, status(distinct_keys == weekly_rows)),
        ('gold_clinical_daily rows', 1400, daily_rows, status(daily_rows == 1400)),
    ],
)
schema = connection.execute(
    '''SELECT column_name, data_type FROM information_schema.columns
       WHERE table_name = 'gold_biostat_weekly' ORDER BY ordinal_position'''
).fetchall()
expected_schema = [(name, kind) for name, kind in WEEKLY_COLUMNS]
render_table(
    'Weekly schema (observed)',
    ['position', 'column', 'type'],
    [(i + 1, name, kind) for i, (name, kind) in enumerate(schema)],
)
daily_columns = {row[0] for row in connection.execute(
    "SELECT column_name FROM information_schema.columns WHERE table_name = 'gold_clinical_daily'").fetchall()}
shared = sorted({name for name, _ in schema} & daily_columns)
show_query(
    'Week boundaries',
    '''SELECT week_id, MIN(period_start_date) AS start_date, MAX(period_end_date) AS end_date,
              COUNT(*) AS participants
       FROM gold_biostat_weekly GROUP BY week_id ORDER BY week_id''',
)
bounds = connection.execute(
    '''SELECT week_id, MIN(period_start_date)::VARCHAR, MAX(period_end_date)::VARCHAR
       FROM gold_biostat_weekly GROUP BY week_id ORDER BY week_id''').fetchall()
expected_bounds = [(1, '2026-01-08', '2026-01-14'), (2, '2026-01-15', '2026-01-21'),
                   (3, '2026-01-22', '2026-01-28'), (4, '2026-01-29', '2026-02-04')]
single_bounds = connection.execute(
    '''SELECT COUNT(*) FROM (SELECT week_id FROM gold_biostat_weekly
       GROUP BY week_id HAVING COUNT(DISTINCT (period_start_date, period_end_date)) > 1)''').fetchone()[0]
render_table(
    'Schema and boundary checks',
    ['check', 'observed', 'status'],
    [
        ('Columns and types equal the specified 29', len(schema), status(schema == expected_schema and len(schema) == 29)),
        ('Columns shared with gold_clinical_daily (only the key columns)', shared, status(set(shared) == {'participant_id'})),
        ('Week bounds are the four study-relative blocks', bounds, status(bounds == expected_bounds)),
        ('Every week has one pair of bounds', single_bounds, status(single_bounds == 0)),
    ],
)
acceptance['200 weekly rows, unique (participant_id, week_id), 4 weeks x 50 participants'] = (
    weekly_rows == 200 and distinct_keys == 200 and enrolled == 50)
acceptance['Weekly schema has the 29 specified columns and differs from the daily schema'] = (
    schema == expected_schema and set(shared) == {'participant_id'})
acceptance['Week bounds are Jan 8-14, 15-21, 22-28 and Jan 29-Feb 4'] = bounds == expected_bounds and single_bounds == 0

check,expected,observed,status
gold_biostat_weekly rows,200,200,PASS
"distinct (participant_id, week_id) keys",200,200,PASS
gold_clinical_daily rows,1400,1400,PASS


position,column,type
1,participant_id,VARCHAR
2,week_id,INTEGER
3,period_start_date,DATE
4,period_end_date,DATE
5,valid_days,INTEGER
6,sleep_valid_days,INTEGER
7,sleep_stage_valid_days,INTEGER
8,steps_valid_days,INTEGER
9,steps_valid_minutes,BIGINT
10,hr_valid_days,INTEGER


week_id,start_date,end_date,participants
1,2026-01-08,2026-01-14,50
2,2026-01-15,2026-01-21,50
3,2026-01-22,2026-01-28,50
4,2026-01-29,2026-02-04,50


check,observed,status
Columns and types equal the specified 29,29,PASS
Columns shared with gold_clinical_daily (only the key columns),['participant_id'],PASS
Week bounds are the four study-relative blocks,"[(1, '2026-01-08', '2026-01-14'), (2, '2026-01-15', '2026-01-21'), (3, '2026-01-22', '2026-01-28'), (4, '2026-01-29', '2026-02-04')]",PASS
Every week has one pair of bounds,0,PASS


In [3]:
# Section 3: coverage columns against the daily table.
connection.execute('''CREATE TEMP VIEW d AS
    SELECT w.participant_id, w.week_id, g.* EXCLUDE (participant_id)
    FROM gold_biostat_weekly w JOIN gold_clinical_daily g
      ON g.participant_id = w.participant_id
     AND g.local_date BETWEEN w.period_start_date AND w.period_end_date''')
show_query(
    'Coverage distribution (rows per valid-day count)',
    '''SELECT 'valid_days' AS column_name, valid_days AS value, COUNT(*) AS rows FROM gold_biostat_weekly GROUP BY ALL
       UNION ALL SELECT 'sleep_valid_days', sleep_valid_days, COUNT(*) FROM gold_biostat_weekly GROUP BY ALL
       UNION ALL SELECT 'steps_valid_days', steps_valid_days, COUNT(*) FROM gold_biostat_weekly GROUP BY ALL
       UNION ALL SELECT 'hr_valid_days', hr_valid_days, COUNT(*) FROM gold_biostat_weekly GROUP BY ALL
       ORDER BY 1, 2''',
)
show_query(
    'Participant-weeks with fewer than 7 steps or heart-rate valid days',
    '''SELECT participant_id, week_id, valid_days, steps_valid_days, steps_valid_minutes,
              hr_valid_days, hr_valid_minutes
       FROM gold_biostat_weekly WHERE steps_valid_days < 7 OR hr_valid_days < 7
       ORDER BY 1, 2''',
)
coverage_mismatch = connection.execute('''
    SELECT COUNT(*) FROM gold_biostat_weekly w JOIN (
        SELECT participant_id, week_id,
               COUNT(total_sleep_min) AS sleep_days, COUNT(deep_sleep_pct) AS stage_days,
               COUNT(total_steps) AS steps_days, COALESCE(SUM(steps_valid_minute_count), 0) AS steps_minutes,
               COUNT(*) FILTER (WHERE hr_valid_minute_count > 0) AS hr_days,
               COALESCE(SUM(hr_valid_minute_count), 0) AS hr_minutes,
               COUNT(*) FILTER (WHERE total_sleep_min IS NOT NULL OR total_steps IS NOT NULL
                                  OR hr_valid_minute_count > 0) AS any_days,
               COUNT(fatigue_score) AS fatigue_days, COUNT(stress_score) AS stress_days,
               COUNT(readiness_score) AS readiness_days, COUNT(sleep_quality_score) AS quality_days
        FROM d GROUP BY 1, 2) x USING (participant_id, week_id)
    WHERE w.sleep_valid_days <> x.sleep_days OR w.sleep_stage_valid_days <> x.stage_days
       OR w.steps_valid_days <> x.steps_days OR w.steps_valid_minutes <> x.steps_minutes
       OR w.hr_valid_days <> x.hr_days OR w.hr_valid_minutes <> x.hr_minutes
       OR w.valid_days <> x.any_days OR w.fatigue_valid_days <> x.fatigue_days
       OR w.stress_valid_days <> x.stress_days OR w.readiness_valid_days <> x.readiness_days
       OR w.sleep_quality_valid_days <> x.quality_days''').fetchone()[0]
null_coverage = connection.execute(
    '''SELECT COUNT(*) FROM gold_biostat_weekly WHERE valid_days IS NULL OR sleep_valid_days IS NULL
       OR steps_valid_days IS NULL OR hr_valid_days IS NULL OR steps_valid_minutes IS NULL
       OR hr_valid_minutes IS NULL''').fetchone()[0]
out_of_range_days = connection.execute(
    'SELECT COUNT(*) FROM gold_biostat_weekly WHERE valid_days NOT BETWEEN 0 AND 7').fetchone()[0]
render_table(
    'Coverage checks',
    ['check', 'observed', 'status'],
    [
        ('Participant-weeks whose coverage columns differ from the daily rows', coverage_mismatch, status(coverage_mismatch == 0)),
        ('Coverage columns that are NULL', null_coverage, status(null_coverage == 0)),
        ('valid_days outside 0 to 7', out_of_range_days, status(out_of_range_days == 0)),
    ],
)
acceptance['Coverage columns equal the counts of valid days and minutes in the daily rows and are never NULL'] = (
    coverage_mismatch == 0 and null_coverage == 0 and out_of_range_days == 0)

column_name,value,rows
hr_valid_days,6,4
hr_valid_days,7,196
sleep_valid_days,7,200
steps_valid_days,6,4
steps_valid_days,7,196
valid_days,7,200


participant_id,week_id,valid_days,steps_valid_days,steps_valid_minutes,hr_valid_days,hr_valid_minutes
P010,1,7,6,7720,6,7347
P019,1,7,6,7741,6,7340
P022,1,7,6,7784,6,7350
P025,1,7,6,7731,6,7351


check,observed,status
Participant-weeks whose coverage columns differ from the daily rows,0,PASS
Coverage columns that are NULL,0,PASS
valid_days outside 0 to 7,0,PASS


In [4]:
# Section 4: weekly values reconciled with gold_clinical_daily (spec section 5, consistency paragraph).
reconciliation = {
    'steps_total = sum of daily total_steps': '''
        SELECT COUNT(*) FROM gold_biostat_weekly w JOIN (
            SELECT participant_id, week_id, SUM(total_steps) AS s FROM d GROUP BY 1, 2) x USING (participant_id, week_id)
        WHERE w.steps_total IS NOT NULL AND w.steps_total IS DISTINCT FROM x.s''',
    'steps_daily_mean = mean of the daily totals': '''
        SELECT COUNT(*) FROM gold_biostat_weekly w JOIN (
            SELECT participant_id, week_id, AVG(total_steps) AS a FROM d GROUP BY 1, 2) x USING (participant_id, week_id)
        WHERE w.steps_daily_mean IS NOT NULL AND ABS(w.steps_daily_mean - x.a) > 1e-9''',
    'rest-to-moderate and burst minutes = sums of the daily counts': '''
        SELECT COUNT(*) FROM gold_biostat_weekly w JOIN (
            SELECT participant_id, week_id, SUM(minutes_rest_to_moderate) AS r, SUM(minutes_burst) AS b
            FROM d GROUP BY 1, 2) x USING (participant_id, week_id)
        WHERE w.steps_total IS NOT NULL
          AND (w.steps_rest_to_moderate_minutes IS DISTINCT FROM x.r OR w.steps_burst_minutes IS DISTINCT FROM x.b)''',
    'hr_weekly_mean_bpm x hr_valid_minutes = sum of daily mean x count': '''
        SELECT COUNT(*) FROM gold_biostat_weekly w JOIN (
            SELECT participant_id, week_id, SUM(hr_mean_bpm * hr_valid_minute_count) AS s FROM d GROUP BY 1, 2) x
            USING (participant_id, week_id)
        WHERE w.hr_weekly_mean_bpm IS NOT NULL AND ABS(w.hr_weekly_mean_bpm * w.hr_valid_minutes - x.s) > 1e-6''',
    'hr min and max = min of daily minima and max of daily maxima': '''
        SELECT COUNT(*) FROM gold_biostat_weekly w JOIN (
            SELECT participant_id, week_id, MIN(hr_min_bpm) AS a, MAX(hr_max_bpm) AS b FROM d GROUP BY 1, 2) x
            USING (participant_id, week_id)
        WHERE w.hr_weekly_mean_bpm IS NOT NULL
          AND (w.hr_weekly_min_bpm IS DISTINCT FROM x.a OR w.hr_weekly_max_bpm IS DISTINCT FROM x.b)''',
    'sleep means = means of the daily values': '''
        SELECT COUNT(*) FROM gold_biostat_weekly w JOIN (
            SELECT participant_id, week_id, AVG(total_sleep_min) AS a, AVG(deep_sleep_pct) AS b,
                   AVG(sleep_efficiency_pct) AS c FROM d GROUP BY 1, 2) x USING (participant_id, week_id)
        WHERE (w.sleep_total_min_mean IS NOT NULL AND ABS(w.sleep_total_min_mean - x.a) > 1e-9)
           OR (w.deep_sleep_pct_mean IS NOT NULL AND ABS(w.deep_sleep_pct_mean - x.b) > 1e-9)
           OR (w.sleep_efficiency_pct_mean IS NOT NULL AND ABS(w.sleep_efficiency_pct_mean - x.c) > 1e-9)''',
    'survey score means = means of the daily scores': '''
        SELECT COUNT(*) FROM gold_biostat_weekly w JOIN (
            SELECT participant_id, week_id, AVG(fatigue_score) AS f, AVG(stress_score) AS s,
                   AVG(readiness_score) AS r, AVG(sleep_quality_score) AS q FROM d GROUP BY 1, 2) x
            USING (participant_id, week_id)
        WHERE (w.fatigue_score_mean IS NOT NULL AND ABS(w.fatigue_score_mean - x.f) > 1e-9)
           OR (w.stress_score_mean IS NOT NULL AND ABS(w.stress_score_mean - x.s) > 1e-9)
           OR (w.readiness_score_mean IS NOT NULL AND ABS(w.readiness_score_mean - x.r) > 1e-9)
           OR (w.sleep_quality_score_mean IS NOT NULL AND ABS(w.sleep_quality_score_mean - x.q) > 1e-9)''',
    'every daily row falls in exactly one week': 'SELECT ABS((SELECT COUNT(*) FROM gold_clinical_daily) - COUNT(*)) FROM d',
}
recon_rows = []
for name, sql in reconciliation.items():
    mismatches = connection.execute(sql).fetchone()[0]
    recon_rows.append((name, mismatches, status(mismatches == 0)))
render_table('Weekly versus daily reconciliation (all 200 participant-weeks)', ['check', 'mismatching rows', 'status'], recon_rows)
show_query(
    'Weekly mean heart rate is minute-weighted, not the mean of daily means (largest differences)',
    '''SELECT w.participant_id, w.week_id, ROUND(w.hr_weekly_mean_bpm, 6) AS weighted_mean,
              ROUND(x.mean_of_means, 6) AS mean_of_daily_means,
              ROUND(ABS(w.hr_weekly_mean_bpm - x.mean_of_means), 6) AS difference
       FROM gold_biostat_weekly w JOIN (
           SELECT participant_id, week_id, AVG(hr_mean_bpm) AS mean_of_means FROM d GROUP BY 1, 2) x
           USING (participant_id, week_id)
       ORDER BY difference DESC, 1, 2 LIMIT 5''',
)
acceptance['Weekly values reconcile with gold_clinical_daily on every participant-week'] = all(
    row[1] == 0 for row in recon_rows)

check,mismatching rows,status
steps_total = sum of daily total_steps,0,PASS
steps_daily_mean = mean of the daily totals,0,PASS
rest-to-moderate and burst minutes = sums of the daily counts,0,PASS
hr_weekly_mean_bpm x hr_valid_minutes = sum of daily mean x count,0,PASS
hr min and max = min of daily minima and max of daily maxima,0,PASS
sleep means = means of the daily values,0,PASS
survey score means = means of the daily scores,0,PASS
every daily row falls in exactly one week,0,PASS


participant_id,week_id,weighted_mean,mean_of_daily_means,difference
P010,1,66.815163,66.760967,0.054195
P019,1,67.644278,67.685979,0.041701
P025,1,71.021358,70.980627,0.040731
P028,2,74.096605,74.068452,0.028153
P041,4,56.145165,56.126356,0.018809


In [5]:
# Section 5: weekly steps and heart-rate values recomputed independently from silver (not from gold).
# One value per minute slot (timestamp truncated to the minute): exact duplicates count once, a conflicting slot is already NULL in silver.
# Rows that are orphan, out of window, undated or flagged folder-mismatch never contribute.
window_start = connection.execute('SELECT study_start FROM run_window').fetchone()[0]
silver_check = connection.execute('''
    WITH steps_slot AS (
        SELECT participant_id, local_date, DATE_TRUNC('minute', minute_ts) AS minute_ts, MIN(steps_per_minute) AS v
        FROM silver_steps_minute
        WHERE NOT is_orphan AND NOT is_out_of_window AND local_date IS NOT NULL
          AND COALESCE(quality_flags, '') NOT LIKE '%FOLDER_MISMATCH%'
        GROUP BY ALL),
    steps_week AS (
        SELECT participant_id, (DATE_DIFF('day', ?::DATE, local_date) // 7) + 1 AS week_id,
               SUM(v) AS total, COUNT(v) AS minutes
        FROM steps_slot GROUP BY 1, 2),
    hr_slot AS (
        SELECT participant_id, local_date, DATE_TRUNC('minute', minute_ts) AS minute_ts, MIN(heart_rate_bpm_normalized) AS v
        FROM silver_heart_rate_minute
        WHERE NOT is_orphan AND NOT is_out_of_window AND local_date IS NOT NULL
          AND COALESCE(quality_flags, '') NOT LIKE '%FOLDER_MISMATCH%'
        GROUP BY ALL),
    hr_week AS (
        SELECT participant_id, (DATE_DIFF('day', ?::DATE, local_date) // 7) + 1 AS week_id,
               SUM(v) AS total, COUNT(v) AS minutes, MIN(v) AS lo, MAX(v) AS hi
        FROM hr_slot GROUP BY 1, 2)
    SELECT
        COUNT(*) FILTER (WHERE w.steps_total IS DISTINCT FROM s.total),
        COUNT(*) FILTER (WHERE w.steps_valid_minutes IS DISTINCT FROM s.minutes),
        COUNT(*) FILTER (WHERE w.hr_valid_minutes IS DISTINCT FROM h.minutes),
        COUNT(*) FILTER (WHERE ABS(w.hr_weekly_mean_bpm - h.total / h.minutes) > 1e-9),
        COUNT(*) FILTER (WHERE w.hr_weekly_min_bpm IS DISTINCT FROM h.lo OR w.hr_weekly_max_bpm IS DISTINCT FROM h.hi),
        COUNT(*)
    FROM gold_biostat_weekly w
    LEFT JOIN steps_week s USING (participant_id, week_id)
    LEFT JOIN hr_week h USING (participant_id, week_id)
    WHERE w.steps_valid_days >= ? AND w.hr_valid_days >= ?''',
    [window_start, window_start, MIN_VALID_DAYS, MIN_VALID_DAYS]).fetchone()
labels = ['steps_total', 'steps_valid_minutes', 'hr_valid_minutes', 'hr_weekly_mean_bpm', 'hr min or max']
render_table(
    'Independent silver recomputation (participant-weeks with both sources available: %d)' % silver_check[5],
    ['weekly column', 'mismatching rows', 'status'],
    [(label, count, status(count == 0)) for label, count in zip(labels, silver_check[:5])],
)
acceptance['Weekly steps and heart-rate values equal an independent recomputation from silver'] = (
    all(count == 0 for count in silver_check[:5]) and silver_check[5] > 0)

weekly column,mismatching rows,status
steps_total,0,PASS
steps_valid_minutes,0,PASS
hr_valid_minutes,0,PASS
hr_weekly_mean_bpm,0,PASS
hr min or max,0,PASS


In [6]:
# Section 6: the availability threshold. A measure is NULL exactly when its own valid-day count is below MIN_VALID_DAYS.
gates = {
    'sleep_total_min_mean': 'sleep_valid_days',
    'deep_sleep_pct_mean': 'sleep_stage_valid_days',
    'sleep_efficiency_pct_mean': 'sleep_stage_valid_days',
    'steps_total': 'steps_valid_days',
    'steps_daily_mean': 'steps_valid_days',
    'steps_rest_to_moderate_minutes': 'steps_valid_days',
    'steps_burst_minutes': 'steps_valid_days',
    'hr_weekly_mean_bpm': 'hr_valid_days',
    'hr_weekly_min_bpm': 'hr_valid_days',
    'hr_weekly_max_bpm': 'hr_valid_days',
    'fatigue_score_mean': 'fatigue_valid_days',
    'stress_score_mean': 'stress_valid_days',
    'readiness_score_mean': 'readiness_valid_days',
    'sleep_quality_score_mean': 'sleep_quality_valid_days',
}
threshold_rows = []
for measure, gate in gates.items():
    wrong = connection.execute(
        f'SELECT COUNT(*) FROM gold_biostat_weekly WHERE ({measure} IS NULL) <> ({gate} < {MIN_VALID_DAYS})'
    ).fetchone()[0]
    null_rows = connection.execute(f'SELECT COUNT(*) FROM gold_biostat_weekly WHERE {measure} IS NULL').fetchone()[0]
    threshold_rows.append((measure, gate, null_rows, wrong, status(wrong == 0)))
render_table(
    f'Availability threshold (MIN_VALID_DAYS = {MIN_VALID_DAYS})',
    ['measure', 'gate column', 'rows with NULL measure', 'rows violating the rule', 'status'],
    threshold_rows,
)
below = connection.execute(
    f'''SELECT COUNT(*) FROM gold_biostat_weekly WHERE sleep_valid_days < {MIN_VALID_DAYS}
        OR steps_valid_days < {MIN_VALID_DAYS} OR hr_valid_days < {MIN_VALID_DAYS}''').fetchone()[0]
print(f'Participant-weeks below the threshold for sleep, steps or heart rate on this dataset: {below}.')
if below == 0:
    print('The threshold is therefore not exercised by the real data; the boundary cases (3 versus 4 days) are covered by the tests.')
acceptance['A measure is NULL exactly when its valid-day count is below 4'] = all(row[3] == 0 for row in threshold_rows)

measure,gate column,rows with NULL measure,rows violating the rule,status
sleep_total_min_mean,sleep_valid_days,0,0,PASS
deep_sleep_pct_mean,sleep_stage_valid_days,0,0,PASS
sleep_efficiency_pct_mean,sleep_stage_valid_days,0,0,PASS
steps_total,steps_valid_days,0,0,PASS
steps_daily_mean,steps_valid_days,0,0,PASS
steps_rest_to_moderate_minutes,steps_valid_days,0,0,PASS
steps_burst_minutes,steps_valid_days,0,0,PASS
hr_weekly_mean_bpm,hr_valid_days,0,0,PASS
hr_weekly_min_bpm,hr_valid_days,0,0,PASS
hr_weekly_max_bpm,hr_valid_days,0,0,PASS


Participant-weeks below the threshold for sleep, steps or heart rate on this dataset: 0.
The threshold is therefore not exercised by the real data; the boundary cases (3 versus 4 days) are covered by the tests.


In [7]:
# Section 7: daily gold columns versus a previous (pre-weekly, INC-005) run, when it exists.
gold_columns = [row[0] for row in connection.execute(
    '''SELECT column_name FROM information_schema.columns
       WHERE table_name = 'gold_clinical_daily' ORDER BY ordinal_position''').fetchall()]
previous_db = PREVIOUS_RUN_DIR / 'wearable.duckdb'
if previous_db.is_file():
    previous = duckdb.connect(str(previous_db), read_only=True)
    try:
        previous_columns = [row[0] for row in previous.execute(
            '''SELECT column_name FROM information_schema.columns
               WHERE table_name = 'gold_clinical_daily' ORDER BY ordinal_position''').fetchall()]
        before = previous.execute('SELECT * FROM gold_clinical_daily ORDER BY ALL').fetchall()
    finally:
        previous.close()
    after = connection.execute('SELECT * FROM gold_clinical_daily ORDER BY ALL').fetchall()
    identical = before == after and previous_columns == gold_columns
    render_table(
        'gold_clinical_daily versus the previous run',
        ['columns before', 'columns after', 'rows before', 'rows after', 'identical', 'status'],
        [(len(previous_columns), len(gold_columns), len(before), len(after), identical, status(identical))],
    )
    acceptance['gold_clinical_daily is identical to the previous run (28 columns, 1,400 rows)'] = identical
else:
    print('Previous run not found at', previous_db, ': comparison NOT RUN.')
    acceptance['gold_clinical_daily is identical to the previous run (28 columns, 1,400 rows)'] = None

columns before,columns after,rows before,rows after,identical,status
28,28,1400,1400,True,PASS


In [8]:
# Section 8: rerun on an isolated copy of the run, then check raw-data immutability.
# The comparison uses the compare command's library function: every table (including the weekly table) and the report bytes.
connection.close()
WORK = Path(tempfile.mkdtemp(prefix='inc007-review-'))
rerun_dir = WORK / 'rerun'
shutil.copytree(RUN_DIR, rerun_dir, ignore=shutil.ignore_patterns('*.wal'))
rerun_result = run_pipeline(DATA_DIR, rerun_dir)
rerun_compare = compare_outputs(RUN_DIR, rerun_dir)
render_table(
    'Same-input rerun',
    ['check', 'result', 'status'],
    [
        ('Every table and the report identical to the supplied run', format_comparison(rerun_compare).strip(), status(rerun_compare.identical)),
        ('Run ID unchanged', rerun_result.run_id, status(rerun_result.run_id == run_id)),
    ],
)
acceptance['Rerun preserves every table (including gold_biostat_weekly), the report and the run id'] = (
    rerun_compare.identical and rerun_result.run_id == run_id)
shutil.rmtree(rerun_dir, ignore_errors=True)

check_db = duckdb.connect(str(DATABASE_PATH), read_only=True)
manifest = {
    row[0]: row[1]
    for row in check_db.execute('SELECT relative_path, sha256 FROM raw_manifest').fetchall()
}
check_db.close()
hash_mismatches = [
    path for path, digest in manifest.items()
    if hashlib.sha256((DATA_DIR / path).read_bytes()).hexdigest() != digest
]
print('Raw files differing from the run manifest SHA-256:', hash_mismatches[:5] or 'none')
acceptance['Raw files match the run manifest hashes'] = not hash_mismatches

check,result,status
Every table and the report identical to the supplied run,identical,PASS
Run ID unchanged,a826c8397cf5ad8e,PASS


Raw files differing from the run manifest SHA-256: none


In [9]:
# Section 9: one corrected record. Copy the data and the supplied run, correct one P001 steps record (+37 steps at
# 2026-01-20T12:00:00, week 2), run incremental mode on the copy and a full rebuild into a second directory.
data_copy = WORK / 'data'
shutil.copytree(DATA_DIR, data_copy)
steps_path = data_copy / 'wearable_events' / 'P001' / 'steps.json'
records = json.loads(steps_path.read_text())
target = next(i for i, record in enumerate(records) if record['timestamp'] == '2026-01-20T12:00:00')
records[target]['steps'] += 37
steps_path.write_text(json.dumps(records))

incremental_dir = WORK / 'incremental'
shutil.copytree(RUN_DIR, incremental_dir, ignore=shutil.ignore_patterns('*.wal'))
incremental_result = run_incremental(data_copy, incremental_dir)
full_dir = WORK / 'full'
full_result = run_pipeline(data_copy, full_dir)
equal_to_full = compare_outputs(incremental_dir, full_dir)

compare_db = duckdb.connect(':memory:')
compare_db.execute("ATTACH '" + str(RUN_DIR / 'wearable.duckdb') + "' AS before_db (READ_ONLY)")
compare_db.execute("ATTACH '" + str(incremental_dir / 'wearable.duckdb') + "' AS after_db (READ_ONLY)")
try:
    changed_weeks = compare_db.execute(
        '''SELECT DISTINCT participant_id, week_id FROM (
               (SELECT * FROM before_db.gold_biostat_weekly EXCEPT ALL SELECT * FROM after_db.gold_biostat_weekly)
               UNION ALL
               (SELECT * FROM after_db.gold_biostat_weekly EXCEPT ALL SELECT * FROM before_db.gold_biostat_weekly)
           ) ORDER BY 1, 2''').fetchall()
    changed_columns = [
        name for name, _ in WEEKLY_COLUMNS[4:]
        if compare_db.execute(
            'SELECT COUNT(*) FROM before_db.gold_biostat_weekly b JOIN after_db.gold_biostat_weekly a '
            'USING (participant_id, week_id) WHERE b.' + name + ' IS DISTINCT FROM a.' + name).fetchone()[0]
    ]
finally:
    compare_db.close()
steps_columns = {'steps_total', 'steps_daily_mean', 'steps_rest_to_moderate_minutes', 'steps_burst_minutes'}
render_table(
    'Corrected record: incremental versus full rebuild',
    ['check', 'observed', 'status'],
    [
        ('Incremental run parsed only the changed file', incremental_result.changed_paths, status(incremental_result.changed_paths == ('wearable_events/P001/steps.json',))),
        ('Incremental output identical to the full rebuild (all tables, report)', format_comparison(equal_to_full).strip(), status(equal_to_full.identical)),
        ('Run id equals the full rebuild run id', incremental_result.run_id, status(incremental_result.run_id == full_result.run_id)),
        ('Exactly one weekly row differs from the supplied run', [(p, w) for p, w in changed_weeks], status(changed_weeks == [('P001', 2)])),
        ('Only the weekly steps measures changed', changed_columns, status(set(changed_columns) <= steps_columns and bool(changed_columns))),
    ],
)
acceptance['Corrected record: incremental equals a full rebuild'] = equal_to_full.identical and incremental_result.run_id == full_result.run_id
acceptance['Corrected record: exactly one weekly row changes (P001, week 2), in the steps measures only'] = (
    changed_weeks == [('P001', 2)] and set(changed_columns) <= steps_columns and bool(changed_columns))
shutil.rmtree(WORK, ignore_errors=True)

check,observed,status
Incremental run parsed only the changed file,"('wearable_events/P001/steps.json',)",PASS
"Incremental output identical to the full rebuild (all tables, report)",identical,PASS
Run id equals the full rebuild run id,41c339cef2ec4025,PASS
Exactly one weekly row differs from the supplied run,"[('P001', 2)]",PASS
Only the weekly steps measures changed,"['steps_total', 'steps_daily_mean', 'steps_rest_to_moderate_minutes', 'steps_burst_minutes']",PASS


In [10]:
# Section 10: acceptance summary.
raw_hash_after = tree_hash(DATA_DIR)
print(f'Raw data fingerprint after: {raw_hash_after[:16]}')
acceptance['Raw data directory unchanged'] = raw_hash_after == RAW_HASH_BEFORE
rows = [
    (name, 'PASS' if result is True else 'FAIL' if result is False else 'NOT RUN')
    for name, result in acceptance.items()
]
rows.append(('Threshold boundary (3 versus 4 days), partial and duplicate-session weeks, window and old-schema cases', 'NOT RUN here; covered by the pipeline tests'))
render_table('INC-007 review summary', ['acceptance check', 'status'], rows)
print('Review is observational; only temporary copies were rerun and they have been removed.')

Raw data fingerprint after: abd71e9575b8ae2a


acceptance check,status
"200 weekly rows, unique (participant_id, week_id), 4 weeks x 50 participants",PASS
Weekly schema has the 29 specified columns and differs from the daily schema,PASS
"Week bounds are Jan 8-14, 15-21, 22-28 and Jan 29-Feb 4",PASS
Coverage columns equal the counts of valid days and minutes in the daily rows and are never NULL,PASS
Weekly values reconcile with gold_clinical_daily on every participant-week,PASS
Weekly steps and heart-rate values equal an independent recomputation from silver,PASS
A measure is NULL exactly when its valid-day count is below 4,PASS
"gold_clinical_daily is identical to the previous run (28 columns, 1,400 rows)",PASS
"Rerun preserves every table (including gold_biostat_weekly), the report and the run id",PASS
Raw files match the run manifest hashes,PASS


Review is observational; only temporary copies were rerun and they have been removed.
